# Treino no Google Colab: detecção de toxicidade no chat de LoL

Roda no Colab as mesmas etapas de `src/train.py` que rodariam localmente:

- **Entrada:** a pasta `PLN/` no seu Google Drive (código, dados, cache dos folds e resultados já prontos).
- **Saída:** resultados e modelos salvos direto em `MyDrive/PLN/PLN/outputs/`.

**Antes de rodar:** *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)*.

Se a sessão cair, rode tudo de novo: os folds já concluídos ficam salvos no Drive e são pulados.

## 1. GPU e Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

## 2. Copiar código e dados para o disco local (rápido); resultados continuam no Drive

In [ ]:
import os
DRIVE_BASE = '/content/drive/MyDrive/PLN'       # pasta PLN enviada para o Drive
LOCAL_BASE = '/content/PLN'

assert os.path.exists(f'{DRIVE_BASE}/PLN/src/train.py'), 'Envie a pasta PLN para a raiz do Meu Drive'
!mkdir -p {LOCAL_BASE}
!rsync -a --exclude outputs {DRIVE_BASE}/ {LOCAL_BASE}/

os.environ['PLN_OUTPUTS'] = f'{DRIVE_BASE}/PLN/outputs'   # resultados e modelos -> Drive
os.environ['PLN_CHECKPOINTS'] = '/content/checkpoints'     # checkpoints temporários -> disco da VM
os.environ['PYTHONIOENCODING'] = 'utf-8'
os.environ['PYTHONUNBUFFERED'] = '1'
os.makedirs(os.environ['PLN_OUTPUTS'], exist_ok=True)
%cd {LOCAL_BASE}/PLN/src

## 3. Dependências (o Colab já traz o PyTorch com CUDA)

In [ ]:
!pip -q install -r ../requirements.txt
import torch, transformers, sklearn
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| transformers', transformers.__version__, '| sklearn', sklearn.__version__)

## 4. Conferir dados e folds

Os folds vêm do cache (`cache/folds_*_k5.csv`), então são exatamente os mesmos do treino local.
Esta célula precisa terminar com `OK: nenhuma partida compartilhada...`.

In [ ]:
!python data.py

## 5. Validação cruzada agrupada por partida (folds prontos são pulados)

In [ ]:
!python train.py --model bert --folds 5 --protocol group

In [ ]:
!python train.py --model roberta --folds 5 --protocol group

## 6. Modelos finais (execução única no split oficial)

Salva o modelo completo em `outputs/<modelo>/best_model/`, junto com uma ficha (`README.md`) com as métricas.

In [ ]:
for m in ['distilbert', 'bert', 'roberta']:
    !python train.py --model {m}

## 7. Avaliação comparativa no teste oficial (métricas por subgrupo e predições)

In [ ]:
!python evaluate.py

## 8. Resumo

In [ ]:
import json, pathlib
out = pathlib.Path(os.environ['PLN_OUTPUTS'])
for m in ['distilbert', 'bert', 'roberta']:
    for p in ['cv_group', 'cv_message']:
        f = out / m / p / 'cv_summary.json'
        if f.exists():
            s = json.loads(f.read_text())
            print(f"{m:10s} {p:10s} F1 = {s['test_mean']['f1']:.4f} ± {s['test_std']['f1']:.4f}")
!du -sh {out}/*/best_model

## 9. Playground: testar mensagens nos três modelos

**Para só usar o playground, sem treinar de novo:** rode as células 1, 2 e 3 e depois pule direto para esta seção.
Ela carrega os modelos de `MyDrive/PLN/PLN/outputs/<modelo>/best_model`.

O Gradio gera um link público (`*.gradio.live`, válido por 72 h) que dá para mandar para os colegas.

**Registro:** cada mensagem avaliada vai para `outputs/playground/avaliacoes.csv`, e cada feedback de erro para
`outputs/playground/feedback.csv`, os dois no Drive. Só o texto e as predições são registrados, sem nome nem IP,
e a página avisa isso a quem testa.

In [ ]:
!pip -q install gradio
import csv
import os
import sys
import threading
import uuid
from datetime import datetime, timezone

sys.path.insert(0, f'{LOCAL_BASE}/PLN/src')
import gradio as gr
import pandas as pd
from inference import load_models

MODELS = load_models(include_baseline=True)   # 3 modelos ajustados + toxic-bert sem fine-tuning
TUNED = [m for m in MODELS if not m.baseline]

# Registro no Drive: uma linha por mensagem avaliada + uma linha por feedback enviado.
LOG_DIR = os.path.join(os.environ['PLN_OUTPUTS'], 'playground')
os.makedirs(LOG_DIR, exist_ok=True)
LOG_PATH = os.path.join(LOG_DIR, 'avaliacoes.csv')
FEEDBACK_PATH = os.path.join(LOG_DIR, 'feedback.csv')
_lock = threading.Lock()


def _append(path, row):
    with _lock:
        new_file = not os.path.exists(path)
        with open(path, 'a', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(row))
            if new_file:
                writer.writeheader()
            writer.writerow(row)


def _now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')


def avaliar(mensagem, limiar, sessao):
    sessao = sessao or uuid.uuid4().hex[:8]   # id aleatório por aba do navegador; não identifica a pessoa
    texts = [t.strip() for t in (mensagem or '').splitlines() if t.strip()]
    if not texts:
        return pd.DataFrame(), 'Digite ao menos uma mensagem.', [], sessao, gr.update(choices=[], value=None)

    probs = {m.key: m.predict_proba(texts) for m in MODELS}
    quando = _now()
    linhas, resumo, avaliadas = [], [], []
    for i, texto in enumerate(texts):
        eval_id = uuid.uuid4().hex[:10]
        votos = sum(probs[m.key][i] >= limiar for m in TUNED)
        registro = {'eval_id': eval_id, 'quando_utc': quando, 'sessao': sessao, 'mensagem': texto,
                    'limiar': limiar}
        for m in MODELS:
            p = probs[m.key][i]
            t = m.default_threshold if m.baseline else limiar
            registro[f'prob_{m.key}'] = round(p, 4)
            linhas.append({'mensagem': texto, 'modelo': m.name, 'prob. tóxica': round(p, 4),
                           'veredito': '⚠ tóxica' if p >= t else '✓ não tóxica'})
        registro['ajustados_toxica'] = f'{votos}/{len(TUNED)}'
        _append(LOG_PATH, registro)
        avaliadas.append([eval_id, texto])
        resumo.append(f'- **{texto}**: {votos} de {len(TUNED)} modelos ajustados consideram tóxica')

    opcoes = [t for _, t in avaliadas]
    return (pd.DataFrame(linhas), '\n'.join(resumo), avaliadas, sessao,
            gr.update(choices=opcoes, value=opcoes[0]))


def registrar_erro(avaliadas, escolhida, correto, comentario, sessao):
    if not avaliadas:
        return 'Avalie uma mensagem antes de enviar o feedback.'
    eval_id, texto = next(((e, t) for e, t in avaliadas if t == escolhida), avaliadas[0])
    _append(FEEDBACK_PATH, {'eval_id': eval_id, 'quando_utc': _now(), 'sessao': sessao or '',
                            'mensagem': texto, 'rotulo_correto': correto,
                            'comentario': (comentario or '').strip()})
    return f'Obrigado! Registrado que "{texto}" deveria ser **{correto}**.'


AVISO = ('**Aviso:** as mensagens testadas e os comentários enviados ficam registrados, sem nome nem IP, '
         'para análise no trabalho final da disciplina de PLN (UFRGS). Não digite dados pessoais.')

with gr.Blocks(title='Toxicidade no chat de LoL') as demo:
    gr.Markdown('# Toxicidade no chat de LoL\n'
                'DistilBERT, BERT e RoBERTa ajustados no dataset L2DTnH, comparados ao toxic-bert '
                'sem fine-tuning (limiar fixo de 0,45). Escreva mensagens de chat em inglês, uma por linha.')
    gr.Markdown(AVISO)
    sessao = gr.State('')
    avaliadas = gr.State([])
    with gr.Row():
        with gr.Column(scale=3):
            entrada = gr.Textbox(lines=4, label='Mensagem(ns)', placeholder='report mid, he is trolling')
        with gr.Column(scale=1):
            limiar = gr.Slider(0.05, 0.95, value=0.5, step=0.05, label='Limiar dos modelos ajustados')
            botao = gr.Button('Avaliar', variant='primary')
    gr.Examples(['gg wp', 'uninstall the game you trash noob', 'STUPID FUCKING LAG',
                 'report jungle he is feeding', 'bg'], inputs=entrada)
    consenso = gr.Markdown()
    tabela = gr.Dataframe(label='Probabilidade por modelo', interactive=False)
    with gr.Accordion('O modelo errou? Conte pra gente', open=False):
        qual = gr.Dropdown(label='Mensagem', choices=[])
        correto = gr.Radio(['tóxica', 'não tóxica'], value='tóxica', label='Na sua opinião, a mensagem é')
        comentario = gr.Textbox(label='Comentário (opcional)')
        enviar = gr.Button('Enviar feedback')
        status = gr.Markdown()

    botao.click(avaliar, [entrada, limiar, sessao], [tabela, consenso, avaliadas, sessao, qual])
    enviar.click(registrar_erro, [avaliadas, qual, correto, comentario, sessao], [status])

demo.queue().launch(share=True)
print('Registro em:', LOG_DIR)

## 10. Ver o que foi testado

Pode rodar a qualquer momento com o playground no ar. Os arquivos também podem ser abertos direto no Drive ou no Google Sheets.

In [ ]:
import os
import pandas as pd

LOG_DIR = os.path.join(os.environ['PLN_OUTPUTS'], 'playground')
log_path, fb_path = os.path.join(LOG_DIR, 'avaliacoes.csv'), os.path.join(LOG_DIR, 'feedback.csv')

if os.path.exists(log_path):
    log = pd.read_csv(log_path)
    print(f"{len(log)} mensagens avaliadas em {log['sessao'].nunique()} sessões")
    display(log.tail(30))
else:
    print('Nenhuma avaliação registrada ainda.')

if os.path.exists(fb_path):
    fb = pd.read_csv(fb_path)
    print(f'\n{len(fb)} feedbacks de erro')
    display(fb.tail(30))